In [ ]:
"""
SI Figures S1-S7 - Paper style
====================================

  S1: Surrogate Fidelity During Extreme Flows (1x3)
  S2: Surrogate Mass Fidelity (1x3)
  S3: Effort-Discharge Scaling (1x2)
  S4: Robustness of Spatial Allocation (1x3)
  S5: Planning Resolution Flexibility (1x3 maps)
  S6: Physical Feasibility (1x3)  [delta-based a,b + original c]
  S7: Stream-Order Controllability (1x3) [representative member]

Requires:
  - outputs_flood_rl2/ (parquet results, includes nse_openloop and rlT_bias_openloop columns)
  - mass_conservation_summary.csv
  - outputs_flood_rl2/delta_atten_per_reach.parquet
  - shape/Flowlines_huc02_12_Brazos.shp, shape/WBDHU12_Brazos.shp
  - HUC comparison output + WBD HUC8/HUC10 shapefiles (paths set in S5 cell)
"""

import warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib as mpl
import matplotlib.pyplot as plt
from scipy.stats import linregress, spearmanr

SHP_PATH   = Path("shape/Flowlines_huc02_12_Brazos.shp")
RESULT_DIR = Path("outputs_flood_rl2")
FIG_DIR    = Path("figures_wrr_new"); FIG_DIR.mkdir(parents=True, exist_ok=True)

GCMS = ["MPI-ESM1-2-HR","ACCESS-CM2","EC-Earth3",
        "CNRM-ESM2-1","BCC-CSM2-MR","MRI-ESM2-0","NorESM2-MM"]
SCENARIOS     = [126, 245, 370, 585]
TARGET_BUDGET = 10.0

SSP_LABELS = {126:"SSP1-2.6", 245:"SSP2-4.5", 370:"SSP3-7.0", 585:"SSP5-8.5"}
SSP_COLORS = {126:"#2d8e3d", 245:"#2166AC", 370:"#d4820a", 585:"#c72e29"}

MM = 1/25.4; W_FULL = 190*MM

# Paper style (NO dark theme)
mpl.rcParams.update({
    "font.family": "Arial", "font.size": 7,
    "axes.labelsize": 7, "axes.titlesize": 7, "axes.linewidth": 0.5,
    "axes.spines.top": False, "axes.spines.right": False,
    "xtick.labelsize": 6, "ytick.labelsize": 6,
    "xtick.major.width": 0.4, "ytick.major.width": 0.4,
    "xtick.major.size": 2.5, "ytick.major.size": 2.5,
    "xtick.direction": "out", "ytick.direction": "out",
    "legend.fontsize": 6, "legend.frameon": False,
    "legend.handlelength": 1.2, "legend.handletextpad": 0.4,
    "figure.dpi": 200, "savefig.dpi": 300,
    "savefig.bbox": "tight", "savefig.pad_inches": 0.02,
    "pdf.fonttype": 42, "ps.fonttype": 42, "lines.linewidth": 0.8,
})

_RA  = 0.12; _MS  = 4.0; _MEW = 0.3; _GA  = 0.08
_PX  = -0.18; _PY  = 1.05
NSE_TH = 0.5
RETURN_PERIODS = [2, 5, 10, 20, 50, 100]

def lgrid(ax):
    ax.grid(True, alpha=_GA, lw=0.3, zorder=0); ax.set_axisbelow(True)

def plabel(ax, lab):
    ax.text(_PX, _PY, lab, transform=ax.transAxes,
            fontweight="bold", fontsize=9, va="top", ha="right")

def weighted_quantile(values, quantiles, weights):
    v = np.asarray(values, dtype=float)
    w = np.asarray(weights, dtype=float)
    m = np.isfinite(v) & np.isfinite(w) & (w > 0)
    v, w = v[m], w[m]
    if len(v) == 0: return np.full_like(quantiles, np.nan, dtype=float)
    s = np.argsort(v); v, w = v[s], w[s]
    cw = np.cumsum(w); cdf = (cw - 0.5*w)/cw[-1]
    return np.interp(quantiles, cdf, v)

def flow_weighted_median(x, w):
    return float(weighted_quantile(x, [0.5], w)[0])

def jitter_strip(ax, x_pos, values, color, width=0.25, ms=3.5, alpha=0.5):
    jit = np.random.default_rng(42).uniform(-width/2, width/2, len(values))
    ax.scatter(x_pos + jit, values, s=ms, alpha=alpha, color=color,
               edgecolors="white", linewidths=0.15, zorder=3, rasterized=True)

def dot_whisker(ax, x_pos, values, color, lw=2.0, ms=7):
    med = np.median(values)
    q25, q75 = np.percentile(values, [25, 75])
    ax.plot([x_pos, x_pos], [q25, q75], color=color, lw=lw,
            solid_capstyle="round", zorder=4)
    ax.plot(x_pos, med, "o", color=color, ms=ms,
            markeredgecolor="black", markeredgewidth=0.5, zorder=5)
    return med

def interpolate_to_budget(grp, target=TARGET_BUDGET):
    eff_by_r = grp.groupby("R_weight")["eff_total_L1"].sum()/1e9
    r_values, efforts = eff_by_r.index.values, eff_by_r.values
    if len(r_values) < 2:
        return grp[grp["R_weight"]==r_values[0]].copy()
    si = np.argsort(efforts); es, rs = efforts[si], r_values[si]
    if target<=es[0]:  return grp[grp["R_weight"]==rs[0]].copy()
    if target>=es[-1]: return grp[grp["R_weight"]==rs[-1]].copy()
    ia = np.searchsorted(es, target); ib = ia-1
    w = 0.5 if es[ia]==es[ib] else (target-es[ib])/(es[ia]-es[ib])
    dlo = grp[grp["R_weight"]==rs[ib]].set_index("reach_id")
    dhi = grp[grp["R_weight"]==rs[ia]].set_index("reach_id")
    common = dlo.index.intersection(dhi.index)
    dlo, dhi = dlo.loc[common], dhi.loc[common]
    out = dlo.copy()
    for c in dlo.select_dtypes(include=[np.number]).columns:
        if c in ("R_weight","scenario","reach_id"): continue
        out[c] = dlo[c]*(1-w)+dhi[c]*w
    return out.reset_index()


# DATA LOADING (memory-streamed: no global df_all to avoid OOM)
import gc
print("Loading data ...")
gdf_net = gpd.read_file(SHP_PATH)
gdf_net["COMID"] = gdf_net["COMID"].astype(int)

# Stream results parquets one at a time and accumulate two small derived
# frames: df_sel (interpolated to TARGET_BUDGET per member) and df_base
# (minimum R_weight per member). The full long-format df_all is NEVER built;
# the original notebook held ~5 GB of parquet in RAM and hit OOM at S4(c).
print(f"  Streaming results parquets and interpolating to {TARGET_BUDGET} Gm3/yr ...")
sel_chunks = []
base_chunks = []
for fp in sorted(RESULT_DIR.glob("results_*.parquet")):
    df = pd.read_parquet(fp)
    if "hydro" not in df.columns:
        df["hydro"] = "VIC5"
    df["reach_id"] = df["reach_id"].astype(int)
    df["scenario"] = df["scenario"].astype(int)

    for k, g in df.groupby(["gcm","scenario","hydro"]):
        sel_chunks.append(interpolate_to_budget(g))

    idx_min = df.groupby(["gcm","hydro","scenario"])["R_weight"].transform("min") == df["R_weight"]
    base_chunks.append(
        df[idx_min].drop_duplicates(subset=["gcm","hydro","scenario","reach_id"]).copy()
    )
    del df, idx_min
    gc.collect()

df_sel = pd.concat(sel_chunks, ignore_index=True)
df_base = pd.concat(base_chunks, ignore_index=True)
del sel_chunks, base_chunks
gc.collect()

mc = [c for c in ["vol_hi_baseline","vol_hi_controlled","eff_total_L1",
                   "residual_ratio","mean_flow_ref"] if c in df_sel.columns]
df_ens = df_sel.groupby(["reach_id","scenario"])[mc].median().reset_index()
df_ens = df_ens.merge(gdf_net[["COMID","StreamOrde","TotDASqKM"]].rename(
    columns={"COMID":"reach_id"}), on="reach_id", how="left")
if "mean_flow_ref" in df_sel.columns and "mean_flow_ref" not in df_ens.columns:
    df_ens["mean_flow_ref"] = df_ens["reach_id"].map(
        df_sel.groupby("reach_id")["mean_flow_ref"].first())
df_ens["so"] = df_ens["StreamOrde"].astype(float).astype("Int64")

daily_frames = [pd.read_parquet(f) for f in sorted(RESULT_DIR.glob("daily_*.parquet"))]
df_daily = pd.concat(daily_frames, ignore_index=True) if daily_frames else None
if df_daily is not None:
    df_daily = df_daily[df_daily["budget_target"]==TARGET_BUDGET]
    df_daily["date"] = pd.to_datetime(df_daily["date"])
    df_daily["year"] = df_daily["date"].dt.year

if "atten_fraction" not in df_base.columns and "eff_atten_L1" in df_base.columns:
    df_base["atten_fraction"] = df_base["eff_atten_L1"]/(df_base["eff_total_L1"]+1e-12)

print(f"  {df_ens['reach_id'].nunique():,} reaches")


In [ ]:
# FIG S1 - Surrogate Fidelity During Extreme Flows (1x3)
#   (a) Per-reach RL10 bias histogram         [from outputs_flood_rl2/ parquet]
#   (b) Basin-aggregate hydrograph RL bias    [from surrogate_fidelity_summary.csv]
#   (c) Stream-order aggregate RL10 bias      [from surrogate_fidelity_summary.csv]
#
# Panel (a) reads the per-reach `rl10_bias_openloop` column directly from the
# step-4 parquets (already loaded as df_base in cell 0). Panels (b) and (c)
# need the basin/order-aggregate AMS, which can only be computed by re-running
# the surrogate (Jensen's inequality on the AMS-RL operator means that the
# flow-weighted reach-average is NOT equal to the basin-aggregate RL bias).
# Those values are pre-computed offline by `si_surrogate_fidelity.py` and
# saved to surrogate_fidelity_summary.csv (one row per member, ~10 KB).
print("\nFIG S1 - Surrogate Fidelity")

RPS = [2, 5, 10, 20, 50, 100]
NSE_TH_VAL = 0.5

df_val = df_base.copy()
if "nse_openloop" in df_val.columns:
    df_val = df_val[df_val["nse_openloop"] > NSE_TH_VAL]
print(f"  (a) {len(df_val):,} reach-member rows (NSE > {NSE_TH_VAL})")

if "StreamOrde" not in df_val.columns:
    df_val = df_val.merge(
        gdf_net[["COMID","StreamOrde"]].rename(columns={"COMID":"reach_id"}),
        on="reach_id", how="left")

fidelity_path = Path("surrogate_fidelity_summary.csv")
if fidelity_path.exists():
    df_fid = pd.read_csv(fidelity_path)
    print(f"  (b,c) {len(df_fid)} members from {fidelity_path}")
else:
    df_fid = None
    print(f"  (b,c) {fidelity_path} not found - run `python si_surrogate_fidelity.py` first")

np.random.seed(42)
fig, axes = plt.subplots(1, 3, figsize=(W_FULL, 65*MM),
                          gridspec_kw={"wspace": 0.38})

# (a) Per-reach RL10 bias histogram
ax = axes[0]
if "rl10_bias_openloop" in df_val.columns:
    bias_arr = pd.to_numeric(df_val["rl10_bias_openloop"], errors="coerce").dropna().values
    bias_arr = bias_arr[np.isfinite(bias_arr)]
    ax.hist(np.clip(bias_arr, -1, 1), bins=80, range=(-1, 1),
            color="#9ecae1", edgecolor="#4292c6", linewidth=0.3,
            density=True, zorder=2)
    med = np.median(bias_arr)
    ax.axvline(med, color="#333", ls="--", lw=0.8, zorder=3)
    ax.axvline(0, color="#999", ls="-", lw=0.4, zorder=1)
    ymax = ax.get_ylim()[1]
    ax.text(med - 0.03, ymax * 0.92,
            f"median\n{med:.3f}", fontsize=5.5, ha="right", va="top")
    ax.set_xlim(-1, 1)
    print(f"  (a) RL10 bias median = {med:.4f}")
ax.set_xlabel(r"RL$_{10}$ relative bias")
ax.set_ylabel("Density")
lgrid(ax); plabel(ax, "a")

# (b) Basin-aggregate hydrograph RL bias by return period
ax = axes[1]
if df_fid is not None and any(f"basin_rl{T}_bias" in df_fid.columns for T in RPS):
    bias_mat = np.array([
        [df_fid[f"basin_rl{T}_bias"].iloc[i] if f"basin_rl{T}_bias" in df_fid.columns else np.nan
         for T in RPS]
        for i in range(len(df_fid))
    ]) * 100  # to %

    log_rps = np.log10(RPS)
    for i in range(bias_mat.shape[0]):
        jitter = np.random.uniform(-0.06, 0.06, len(RPS))
        x_jit = 10 ** (log_rps + jitter)
        ax.scatter(x_jit, bias_mat[i, :],
                   s=5, color="#9ecae1", alpha=0.35, zorder=1,
                   edgecolors="none")
    med_line = np.nanmedian(bias_mat, axis=0)
    ax.plot(RPS, med_line, "o-", color="#d62728",
            ms=5, lw=1.0, zorder=3, label="Ensemble median")
    ax.axhline(0, color="#999", ls="-", lw=0.4, zorder=0)
    ax.axhline(5, color="#999", ls=":", lw=0.5, zorder=0)
    ax.axhline(-5, color="#999", ls=":", lw=0.5, zorder=0)
    ax.set_ylim(-10, 10)
    ax.text(0.97, 0.05, f"n = {bias_mat.shape[0]} members",
            transform=ax.transAxes, fontsize=5, va="bottom", ha="right",
            color="#555")
    print(f"  (b) median basin RL10 bias = {med_line[2]:+.2f}%")
else:
    ax.text(0.5, 0.5,
            "surrogate_fidelity_summary.csv\nnot found.\n\n"
            "Run `python si_surrogate_fidelity.py`\n"
            "to generate it.",
            transform=ax.transAxes, ha="center", va="center",
            fontsize=6, color="#666")

ax.set_xlabel("Return period (years)")
ax.set_ylabel("Basin-total RL bias (%)")
ax.set_xscale("log")
ax.set_xticks(RPS)
ax.set_xticklabels([str(r) for r in RPS])
ax.legend(loc="lower left", fontsize=5)
lgrid(ax); plabel(ax, "b")

# (c) Stream-order aggregate RL10 bias
ax = axes[2]
if df_fid is not None:
    order_cols = sorted(
        [c for c in df_fid.columns if c.startswith("order") and c.endswith("_rl10_bias")],
        key=lambda c: int(c.replace("order", "").replace("_rl10_bias", ""))
    )
    if order_cols:
        orders = [int(c.replace("order", "").replace("_rl10_bias", "")) for c in order_cols]
        order_mat = df_fid[order_cols].values * 100  # to %

        x_pos = np.arange(len(orders))
        for i in range(order_mat.shape[0]):
            jitter = np.random.uniform(-0.15, 0.15, len(orders))
            ax.scatter(x_pos + jitter, order_mat[i, :],
                       s=5, color="#9ecae1", alpha=0.35, zorder=1,
                       edgecolors="none")
        med_o = np.nanmedian(order_mat, axis=0)
        ax.plot(x_pos, med_o, "o-", color="#d62728",
                ms=5, lw=1.0, zorder=3, label="Ensemble median")
        ax.axhline(0, color="#999", ls="-", lw=0.4, zorder=0)
        ax.axhline(5, color="#999", ls=":", lw=0.5, zorder=0)
        ax.axhline(-5, color="#999", ls=":", lw=0.5, zorder=0)
        ax.set_xticks(x_pos)
        ax.set_xticklabels([str(o) for o in orders])
        ax.legend(loc="upper right", fontsize=5)
        print(f"  (c) {len(orders)} stream orders")
else:
    ax.text(0.5, 0.5, "surrogate_fidelity_summary.csv\nnot found",
            transform=ax.transAxes, ha="center", va="center",
            fontsize=6, color="#666")

ax.set_ylim(-10, 10)
ax.set_xlabel("Stream order")
ax.set_ylabel(r"RL$_{10}$ bias (%)")
lgrid(ax); plabel(ax, "c")

for ext in ("png", "pdf"):
    fig.savefig(FIG_DIR / f"figS1_validation.{ext}")
plt.show()
print("  Saved figS1_validation")


In [ ]:
# FIG S2 - Surrogate Mass Fidelity (1x3)
print("\nFIG S2 - Surrogate Mass Fidelity")

df_mass = pd.read_csv("mass_conservation_summary.csv")
print(f"  Loaded {len(df_mass)} members")
scenarios_s2 = sorted(df_mass["ssp"].unique())

fig, axes = plt.subplots(1, 3, figsize=(W_FULL, 70*MM),
                          gridspec_kw={"wspace": 0.36})

# (a) Volume ratio
ax = axes[0]
ax.axhline(1.0, color="#777", ls="--", lw=0.6, zorder=1)
for i, scn in enumerate(scenarios_s2):
    vals = df_mass[df_mass["ssp"]==scn]["vol_ratio"].dropna().values
    jitter_strip(ax, i, vals, SSP_COLORS[scn], ms=4, alpha=0.55)
    dot_whisker(ax, i, vals, SSP_COLORS[scn])
all_vr = df_mass["vol_ratio"].dropna().values
ax.text(0.97, 0.05, f"all: {np.median(all_vr):.3f}\n(n={len(all_vr)})",
        transform=ax.transAxes, fontsize=6.5, va="bottom", ha="right",
        bbox=dict(boxstyle="round,pad=0.15", fc="white",
                  ec="none", lw=0.3, alpha=0.85))
ax.set_xticks(range(len(scenarios_s2)))
ax.set_xticklabels([SSP_LABELS[s] for s in scenarios_s2], fontsize=5.5)
ax.set_ylabel("Volume ratio\n(surrogate / reference)")
ax.set_xlim(-0.5, len(scenarios_s2)-0.5)
ax.set_ylim(0.95, 1.05)
lgrid(ax); plabel(ax, "a")

# (b) Basin-total delta correlation r
ax = axes[1]
ax.axhline(1.0, color="#777", ls="--", lw=0.6, zorder=1)
for i, scn in enumerate(scenarios_s2):
    vals = df_mass[df_mass["ssp"]==scn]["delta_corr"].dropna().values
    jitter_strip(ax, i, vals, SSP_COLORS[scn], ms=4, alpha=0.55)
    dot_whisker(ax, i, vals, SSP_COLORS[scn])
all_r  = df_mass["delta_corr"].dropna().values
ax.text(0.97, 0.05, f"all: {np.median(all_r):.3f}\n(n={len(all_r)})",
        transform=ax.transAxes, fontsize=6.5, va="bottom", ha="right",
        bbox=dict(boxstyle="round,pad=0.15", fc="white",
                  ec="none", lw=0.3, alpha=0.85))
ax.set_xticks(range(len(scenarios_s2)))
ax.set_xticklabels([SSP_LABELS[s] for s in scenarios_s2], fontsize=5.5)
ax.set_ylabel(r"Basin-total $\delta$ correlation")
ax.set_xlim(-0.5, len(scenarios_s2)-0.5)
ax.set_ylim(-0.05, 1.05)
lgrid(ax); plabel(ax, "b")

# (c) Per-reach delta correlation median
ax = axes[2]
ax.axhline(1.0, color="#777", ls="--", lw=0.6, zorder=1)
for i, scn in enumerate(scenarios_s2):
    vals = df_mass[df_mass["ssp"]==scn]["reach_median"].dropna().values
    jitter_strip(ax, i, vals, SSP_COLORS[scn], ms=4, alpha=0.55)
    dot_whisker(ax, i, vals, SSP_COLORS[scn])
all_rm = df_mass["reach_median"].dropna().values
ax.text(0.97, 0.05, f"all: {np.median(all_rm):.3f}\n(n={len(all_rm)})",
        transform=ax.transAxes, fontsize=6.5, va="bottom", ha="right",
        bbox=dict(boxstyle="round,pad=0.15", fc="white",
                  ec="none", lw=0.3, alpha=0.85))
ax.set_xticks(range(len(scenarios_s2)))
ax.set_xticklabels([SSP_LABELS[s] for s in scenarios_s2], fontsize=5.5)
ax.set_ylabel("Per-reach $\\delta$ correlation\n(median across reaches)")
ax.set_xlim(-0.5, len(scenarios_s2)-0.5)
ax.set_ylim(-0.05, 1.05)
lgrid(ax); plabel(ax, "c")

for ext in ("png","pdf"): fig.savefig(FIG_DIR/f"figS2_mass_fidelity.{ext}")
plt.show()


In [ ]:
# FIG S3 - Effort-Discharge Scaling (1x2)
print("\nFIG S3 - Regression")
sub = df_ens[df_ens["scenario"]==245].copy()
eff = sub["eff_total_L1"].to_numpy(float)
da  = sub["TotDASqKM"].to_numpy(float)
mf  = sub["mean_flow_ref"].to_numpy(float) if "mean_flow_ref" in sub.columns else None

fig, axes = plt.subplots(1, 2, figsize=(W_FULL, 70*MM),
                          gridspec_kw={"wspace":0.28})

panels = [
    (axes[0], mf,  "Blues",
     r"Mean annual flow $\overline{Q}$ (log$_{10}$ m$^3$ s$^{-1}$)", "a"),
    (axes[1], da,  "Greens",
     r"Drainage area (log$_{10}$ km$^2$)", "b"),
]

for ax, xv, cmap_name, xlab, lab in panels:
    if xv is None:
        ax.text(0.5,0.5,"Data not available",transform=ax.transAxes,ha="center")
        plabel(ax,lab); continue

    m = (eff>0)&(xv>0)&np.isfinite(eff)&np.isfinite(xv)
    lx, ly = np.log10(xv[m]), np.log10(eff[m])
    lr = linregress(lx, ly)

    ax.hexbin(lx, ly, gridsize=45, cmap=cmap_name, mincnt=1,
              linewidths=0, alpha=0.75, zorder=1)

    xf = np.array([lx.min(), lx.max()])
    ax.plot(xf, lr.slope*xf+lr.intercept, "--", color="#333",
            lw=1.3, zorder=5)

    ax.text(0.05, 0.95,
            f"slope = {lr.slope:.2f}\n$R^2$ = {lr.rvalue**2:.2f}",
            transform=ax.transAxes, fontsize=6.5, va="top", ha="left",
            bbox=dict(boxstyle="round,pad=0.2", fc="white",
                      ec="none", lw=0.4, alpha=0.92))

    ax.set_xlabel(xlab)
    ax.set_ylim(-1, 8)
    if lab=="a": ax.set_ylabel(r"Annual effort (log$_{10}$ m$^3$ yr$^{-1}$)")
    lgrid(ax); plabel(ax, lab)
    print(f"  ({lab}) R2={lr.rvalue**2:.3f}")

for ext in ("png","pdf"): fig.savefig(FIG_DIR/f"figS3_regression.{ext}")
plt.show()


In [ ]:
# FIG S4 - Robustness (1x3)
print("\nFIG S4 - Robustness")
so_groups = [("1-2",[1,2]), ("3-4",[3,4]), ("5+",[5,6,7,8,9])]

fig, axes = plt.subplots(1, 3, figsize=(W_FULL, 70*MM),
                          gridspec_kw={"wspace":0.38})

# (a) Stream-order effort
ax = axes[0]
x_pos = np.arange(len(so_groups))
bw = 0.18
for i, scn in enumerate(SCENARIOS):
    sub = df_ens[df_ens["scenario"]==scn]
    total = sub["eff_total_L1"].sum()
    fracs = [sub[sub["so"].isin(o)]["eff_total_L1"].sum()/total*100
             for _,o in so_groups]
    ax.bar(x_pos+(i-1.5)*bw, fracs, bw,
           color=SSP_COLORS[scn], alpha=0.75, edgecolor="black",
           linewidth=0.3, label=SSP_LABELS[scn])

ax.set_xticks(x_pos)
ax.set_xticklabels([g for g,_ in so_groups])
ax.set_xlabel("Stream order")
ax.set_ylabel("Share of basin effort (%)")
ax.set_ylim(0, 58)
ax.legend(fontsize=4.5, loc="upper left", ncol=2,
          handlelength=0.8, columnspacing=0.5)
lgrid(ax); plabel(ax,"a")

for scn in SCENARIOS:
    sub = df_ens[df_ens["scenario"]==scn]
    t = sub["eff_total_L1"].sum()
    o5 = sub[sub["so"].isin([5,6,7,8,9])]["eff_total_L1"].sum()/t*100
    print(f"  (a) {SSP_LABELS[scn]}: Order 5+ = {o5:.1f}%")

# (b) Temporal late/early ratio
ax = axes[1]
if df_daily is not None:
    df_daily["period"] = np.where(df_daily["year"]<2060, "early", "late")
    grp = ["scenario","gcm"] + (["hydro"] if "hydro" in df_daily.columns else [])
    merge_on = ["gcm"] + (["hydro"] if "hydro" in df_daily.columns else [])

    pb = (df_daily.groupby(grp+["period"])
          .agg(cl=("basin_exceed_controlled","sum")).reset_index())
    pb["cl_Gm3"] = pb["cl"]/1e9

    for i, scn in enumerate(SCENARIOS):
        e = pb[(pb["scenario"]==scn)&(pb["period"]=="early")].set_index(merge_on)["cl_Gm3"]
        l = pb[(pb["scenario"]==scn)&(pb["period"]=="late")].set_index(merge_on)["cl_Gm3"]
        c = e.index.intersection(l.index)
        vals = (l.loc[c]/(e.loc[c]+1e-12)).values

        med  = np.median(vals)
        q25, q75 = np.percentile(vals, [25, 75])
        ax.plot([i, i], [q25, q75], color=SSP_COLORS[scn], lw=2.0,
                solid_capstyle="round", zorder=3)
        ax.plot(i, med, "o", color=SSP_COLORS[scn], ms=7,
                markeredgecolor="black", markeredgewidth=0.6, zorder=4)
        ax.text(i, q75+0.08, f"{med:.2f}", ha="center", fontsize=5.5,
                fontweight="bold")
        print(f"  (b) {SSP_LABELS[scn]}: {med:.2f}")

    ax.axhline(1.0, color="#777", ls="--", lw=0.6, zorder=0)
    ax.set_xticks(range(len(SCENARIOS)))
    ax.set_xticklabels([SSP_LABELS[s] for s in SCENARIOS], fontsize=5.5)
    ax.set_ylabel("Late / early century\nexceedance ratio")
    ax.set_xlim(-0.5, len(SCENARIOS)-0.5)
    ax.set_ylim(0.4, 1.8)
else:
    ax.text(0.5,0.5,"Daily data\nnot available",transform=ax.transAxes,
            ha="center",fontsize=7)
lgrid(ax); plabel(ax,"b")

# (c) Ranking stability  [pre-computed by si_ranking_stability.py]
# This used to iterate over the full df_all (~5 GB) and OOM the kernel.
# The script now writes a tiny ranking_stability.csv (one row per member).
ax = axes[2]
ranking_path = Path("ranking_stability.csv")
if ranking_path.exists():
    C = pd.read_csv(ranking_path)
    C["scenario"] = C["scenario"].astype(int)
    print(f"  (c) {len(C)} members from {ranking_path}")
else:
    C = pd.DataFrame()
    print(f"  (c) {ranking_path} not found - run `python si_ranking_stability.py`")

if len(C) > 0:
    offset = 0.12
    for i, scn in enumerate(SCENARIOS):
        cs = C[C["scenario"]==scn]
        for j, (metric, color, marker, label) in enumerate([
            ("spearman", "#4292c6", "o", r"Spearman $\rho$"),
            ("overlap",  "#E6550D", "s", "Top-10% overlap"),
        ]):
            vals = cs[metric].values
            med  = np.median(vals)
            q25, q75 = np.percentile(vals, [25, 75])
            xpos = i + (j-0.5)*offset*2
            ax.plot([xpos, xpos], [q25, q75], color=color, lw=1.5,
                    solid_capstyle="round", zorder=3)
            ax.plot(xpos, med, marker, color=color, ms=6,
                    markeredgecolor="black", markeredgewidth=0.5,
                    zorder=4, label=label if i==0 else "")

    all_v = np.concatenate([C["spearman"].values, C["overlap"].values])
    ax.set_ylim(0.5, 1.02)
    ax.set_xticks(range(len(SCENARIOS)))
    ax.set_xticklabels([SSP_LABELS[s] for s in SCENARIOS], fontsize=5.5)
    ax.set_ylabel("Ranking stability")
    ax.set_xlim(-0.5, len(SCENARIOS)-0.5)
    ax.legend(loc="lower left", fontsize=5, handlelength=1.0)
    print(f"  (c) rho={C['spearman'].median():.3f}, "
          f"overlap={C['overlap'].median():.2f}")

lgrid(ax); plabel(ax,"c")

for ext in ("png","pdf"): fig.savefig(FIG_DIR/f"figS4_robustness.{ext}")
plt.show()


In [ ]:
# FIG S5 - Planning Resolution Flexibility (1x3 maps)
#   (a) HUC12-aggregated effort placement
#   (b) HUC10-aggregated effort placement
#   (c) HUC8-aggregated effort placement
# Source: si_huc_comparison.py outputs (huc_spatial_compact.csv) + WBD shapefiles
print("\nFIG S5 - Planning Resolution Flexibility")

import matplotlib.colors as mcolors
from matplotlib import cm
from matplotlib.collections import LineCollection

# Map helpers (inlined so removing this cell doesn't break others)
def _segs(geom):
    if geom is None or geom.is_empty: return []
    if geom.geom_type == "LineString": return [np.asarray(geom.coords)]
    if geom.geom_type == "MultiLineString":
        return [np.asarray(g.coords) for g in geom.geoms]
    return []

def draw_net_cmap(ax, gdf, values, cmap, norm, lw, alpha=0.92):
    segs, cols, lws = [], [], []
    for geom, v, w in zip(gdf.geometry.values, values, lw):
        for part in _segs(geom):
            segs.append(part)
            cols.append(cmap(norm(v)) if np.isfinite(v) else (0.85,0.85,0.85,0.3))
            lws.append(float(w))
    lc = LineCollection(segs, colors=cols, linewidths=lws,
                        capstyle="round", joinstyle="round", alpha=alpha, zorder=1)
    lc.set_rasterized(True); ax.add_collection(lc)

def add_bg(ax, gdf):
    gdf.plot(ax=ax, color="#f0f0f0", linewidth=0.02, alpha=0.22, zorder=0)

def add_scalebar(ax, gdf, km=100):
    b = gdf.total_bounds; bar = km / 96.0
    x0 = b[2] - bar - (b[2]-b[0])*0.04
    y0 = b[1] + (b[3]-b[1])*0.04
    ax.plot([x0, x0+bar], [y0, y0], "k-", lw=0.8, solid_capstyle="butt", zorder=10)
    ax.text(x0+bar/2, y0+(b[3]-b[1])*0.015, f"{km} km",
            ha="center", fontsize=5, zorder=10)

def tcmap(name, lo=0.0, hi=1.0, n=256):
    base = cm.get_cmap(name)
    return mcolors.LinearSegmentedColormap.from_list(
        f"{name}_t", base(np.linspace(lo, hi, n)))

# Input paths (all relative to the archive root):
#   - huc_spatial_compact.csv     : output of si_huc_comparison.py (concat across members)
#   - comid_huc_mapping_correct.csv : COMID -> HUC8/HUC10/HUC12 mapping (WBD spatial join)
#   - shape/WBDHU{8,10,12}_Brazos.shp : WBD subsetted to the Brazos basin
#     (preferred). Falls back to nationwide shape/WBDHU{8,10}.shp if Brazos
#     subset is missing. WBDHU12_Brazos.shp ships with the archive.
HUC_COMPACT = "huc_spatial_compact.csv"
HUC_MAPPING = "comid_huc_mapping_correct.csv"
WBD_PATHS = {
    "HUC12": ["shape/WBDHU12_Brazos.shp", "shape/WBDHU12.shp"],
    "HUC10": ["shape/WBDHU10_Brazos.shp", "shape/WBDHU10.shp"],
    "HUC8":  ["shape/WBDHU8_Brazos.shp",  "shape/WBDHU8.shp"],
}

df_huc = pd.read_csv(HUC_COMPACT)
huc_map = pd.read_csv(HUC_MAPPING)
huc_map["COMID"] = huc_map["COMID"].astype(int)

df_huc = df_huc.drop(columns=["HUC8","HUC10","HUC12"], errors="ignore")
df_huc = df_huc.merge(huc_map[["COMID","HUC8","HUC10","HUC12"]],
                       left_on="reach_id", right_on="COMID",
                       how="left").drop(columns=["COMID"], errors="ignore")
for col in ["HUC8","HUC10","HUC12"]:
    df_huc[col] = df_huc[col].dropna().astype(float).astype(np.int64).astype(str)

brazos_hucs = {lv: set(df_huc[lv].dropna().unique()) for lv in ["HUC8","HUC10","HUC12"]}

for lv in ["HUC12","HUC10","HUC8"]:
    df_huc[f"diff_{lv}"] = df_huc["peak_red_reach"] - df_huc[f"peak_red_{lv}"]

gdf_huc = gdf_net.merge(df_huc, left_on="COMID", right_on="reach_id", how="inner")
da_col = "TotDASqKM_x" if "TotDASqKM_x" in gdf_huc.columns else "TotDASqKM"
gdf_huc = gdf_huc.sort_values(da_col, ascending=True)

wbd_huc = {}
for lv, candidates in WBD_PATHS.items():
    found_path = next((p for p in candidates if Path(p).exists()), None)
    if found_path is None:
        print(f"  WBD {lv}: not found (looked for {candidates})")
        wbd_huc[lv] = None
        continue
    try:
        w = gpd.read_file(found_path)
        for c in w.columns:
            if c.upper() == lv.upper():
                w = w.rename(columns={c: lv}); break
        w[lv] = w[lv].astype(str).str.strip()
        w = w[w[lv].isin(brazos_hucs[lv])].copy()
        w = w[~w.geometry.is_empty & w.geometry.notna()].copy()
        if gdf_net.crs is not None and w.crs is not None and w.crs != gdf_net.crs:
            w = w.to_crs(gdf_net.crs)
        wbd_huc[lv] = w
        print(f"  WBD {lv}: {len(w)} polygons (from {found_path})")
    except Exception as e:
        print(f"  WBD {lv}: FAILED - {e}")
        wbd_huc[lv] = None

# Map extent with a small pad
_bnds = gdf_net.total_bounds
_xp = (_bnds[2]-_bnds[0])*0.02; _yp = (_bnds[3]-_bnds[1])*0.02
_mxlim = (_bnds[0]-_xp, _bnds[2]+_xp)
_mylim = (_bnds[1]-_yp, _bnds[3]+_yp)

fig_s = plt.figure(figsize=(W_FULL, 85*MM))
gs_s = fig_s.add_gridspec(1, 3, left=0.01, right=0.89, top=0.92, bottom=0.14,
                           wspace=0.03)

cmap_diff = tcmap("RdBu_r", 0.05, 0.95)
diff_lim = 0.50
norm_diff = mcolors.TwoSlopeNorm(vcenter=0, vmin=-diff_lim, vmax=diff_lim)

_panels = [
    ("diff_HUC12", "a", "HUC12 (208 outlets)", "HUC12", 0.3, "99.9%"),
    ("diff_HUC10", "b", "HUC10 (42 outlets)",  "HUC10", 0.6, "98.4%"),
    ("diff_HUC8",  "c", "HUC8 (28 outlets)",   "HUC8",  1.0, "88.6%"),
]

for i, (col, lab, subtitle, bnd_lv, bnd_lw, pct) in enumerate(_panels):
    ax = fig_s.add_subplot(gs_s[0, i])
    ax.set_aspect("equal"); ax.axis("off")
    add_bg(ax, gdf_net)

    gdf_p = gdf_huc.sort_values(col, key=lambda s: s.abs(), ascending=True)
    lw_p = np.full(len(gdf_p), 0.95)
    draw_net_cmap(ax, gdf_p, gdf_p[col].values, cmap_diff, norm_diff, lw_p)

    w = wbd_huc.get(bnd_lv)
    if w is not None and len(w) > 0:
        w.boundary.plot(ax=ax, color="#555555", linewidth=bnd_lw,
                        alpha=0.45, zorder=5)

    ax.set_xlim(_mxlim); ax.set_ylim(_mylim)

    ax.text(-0.05, 0.97, lab, transform=ax.transAxes,
            fontweight="bold", fontsize=11, va="top", ha="left",
            bbox=dict(boxstyle="round,pad=0.12", fc="white", ec="none", alpha=0.9))

    ax.text(0.50, 0.02, subtitle, transform=ax.transAxes, ha="center",
            fontsize=7, fontweight="bold", color="#444",
            bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.9))

    ax.text(0.50, -0.04, f"{pct} of reach-level baseline", transform=ax.transAxes,
            ha="center", fontsize=6.5, color="#666", style="italic")

    if i == 0: add_scalebar(ax, gdf_net)

cax_s = fig_s.add_axes([0.91, 0.18, 0.013, 0.65])
sm_s = cm.ScalarMappable(cmap=cmap_diff, norm=norm_diff)
cb_s = fig_s.colorbar(sm_s, cax=cax_s)
cb_s.set_label(r"$\Delta$ peak reduction vs. reach-level (pp)",
               fontsize=7, labelpad=8)
_dt = [-diff_lim, -diff_lim/2, 0, diff_lim/2, diff_lim]
cb_s.set_ticks(_dt)
cb_s.set_ticklabels([f"{t*100:+.0f}" for t in _dt])
cb_s.ax.tick_params(labelsize=6.5, width=0.3, length=2)
cb_s.outline.set_linewidth(0.3)

for ext in ("png","pdf"):
    fig_s.savefig(FIG_DIR / f"figS5_huc_flexibility.{ext}")
plt.show()
print("  Saved figS5_huc_flexibility")


In [ ]:
# FIG S6 - Physical Feasibility (1x3) [delta-based a,b]
print("\nFIG S6 - Feasibility")

DELTA_DIR = Path("outputs_flood_rl2")
delta_parquet = DELTA_DIR / "delta_atten_per_reach.parquet"
has_delta = delta_parquet.exists()
has_feas  = "feas_median_mean_ratio_flow" in df_mass.columns

if has_delta:
    df_delta = pd.read_parquet(delta_parquet)
    print(f"  Loaded delta data: {len(df_delta):,} rows")

fig, axes = plt.subplots(1, 3, figsize=(W_FULL, 70*MM),
                          gridspec_kw={"wspace": 0.34})

# (a) Histogram: delta atten fraction during exceedance
ax = axes[0]
if has_delta:
    v = df_delta["delta_atten_flood_frac"].dropna().values
    v = v[np.isfinite(v) & (v >= 0) & (v <= 1)] * 100

    ax.hist(v, bins=60, density=True, color="#9ecae1", alpha=0.85,
            edgecolor="#4292c6", linewidth=0.3, zorder=2)

    med = np.median(v)
    p5  = np.percentile(v, 5)
    ax.axvline(med, color="#333", ls="--", lw=0.9, zorder=5)
    ax.axvline(p5,  color="#333", ls=":",  lw=0.7, zorder=5)
    yl = ax.get_ylim()[1]
    ax.text(med - 4, yl * 0.93, f"median\n{med:.0f}%", fontsize=6,
            fontweight="bold", ha="right", va="top")
    ax.text(p5 - 4,  yl * 0.65, f"P5\n{p5:.0f}%", fontsize=6, ha="right", va="top")

    ax.set_xlabel(r"Attenuation fraction $|\delta_i^-|/|\delta_i|$"
                  "\nduring exceedance (%)")
    ax.set_ylabel("Density"); ax.set_xlim(0, 105)
    print(f"  (a) delta flood atten: median={med:.1f}%, P5={p5:.1f}%")
else:
    ax.text(0.5, 0.5, "delta data not available\nRun compute_delta_atten_fraction.py",
            transform=ax.transAxes, ha="center", fontsize=6)
lgrid(ax); plabel(ax, "a")

# (b) Basin-scale delta attenuation per SSP
ax = axes[1]
if has_delta:
    rows_b = []
    for (gcm, hydro, ssp), g in df_delta.groupby(["gcm", "hydro", "ssp"]):
        atten = g["delta_atten_L1"].sum()
        total = g["delta_total_L1"].sum()
        rows_b.append({"ssp": int(ssp), "val": atten / (total + 1e-12) * 100})
    dfc = pd.DataFrame(rows_b)

    for i, scn in enumerate(SCENARIOS):
        vals = dfc[dfc["ssp"] == scn]["val"].values
        if len(vals) == 0: continue
        jitter_strip(ax, i, vals, SSP_COLORS[scn], ms=4, alpha=0.55)
        med = np.median(vals)
        q25, q75 = np.percentile(vals, [25, 75])
        ax.plot([i, i], [q25, q75], color=SSP_COLORS[scn], lw=2.0,
                solid_capstyle="round", zorder=4)
        ax.plot(i, med, "o", color=SSP_COLORS[scn], ms=7,
                markeredgecolor="black", markeredgewidth=0.5, zorder=5)
        ax.text(i, q25 - 4, f"{med:.1f}%", ha="center", fontsize=5.5,
                fontweight="bold", va="top")
        print(f"  (b) {SSP_LABELS[scn]}: {med:.1f}% [{q25:.1f}-{q75:.1f}]")

    all_vr = dfc["val"].values
    ax.text(0.97, 0.05, f"all: {np.median(all_vr):.1f}%\n(n={len(all_vr)})",
            transform=ax.transAxes, fontsize=6.5, va="bottom", ha="right",
            bbox=dict(boxstyle="round,pad=0.15", fc="white",
                      ec="none", lw=0.3, alpha=0.85))
    ax.set_xticks(range(len(SCENARIOS)))
    ax.set_xticklabels([SSP_LABELS[s] for s in SCENARIOS], fontsize=6)
    ax.set_ylabel("Basin-scale attenuation\nfraction (%)")
    ax.set_xlim(-0.5, len(SCENARIOS)-0.5)
    ax.set_ylim(0, 102)
else:
    ax.text(0.5, 0.5, "delta data not available",
            transform=ax.transAxes, ha="center", fontsize=6)
lgrid(ax); plabel(ax, "b")

# (c) |u_i|/x_i during exceedance (median only)
ax = axes[2]
if has_feas:
    for i, scn in enumerate(SCENARIOS):
        sub = df_mass[df_mass["ssp"]==scn]
        vals_med = sub["feas_median_mean_ratio_flow"].dropna().values * 100

        med  = np.median(vals_med)
        q25, q75 = np.percentile(vals_med, [25, 75])
        ax.plot([i, i], [q25, q75], color=SSP_COLORS[scn], lw=2.0,
                solid_capstyle="round", zorder=3)
        ax.plot(i, med, "o", color=SSP_COLORS[scn], ms=7,
                markeredgecolor="black", markeredgewidth=0.6, zorder=4)
        ax.text(i, q75 + 2.0, f"{med:.0f}%", ha="center", fontsize=5.5,
                fontweight="bold")
        print(f"  (c) {SSP_LABELS[scn]}: median={med:.1f}%")

    all_med = df_mass["feas_median_mean_ratio_flow"].dropna().values * 100
    ax.text(0.97, 0.95,
            f"all: median={np.median(all_med):.0f}%\n(n={len(all_med)})",
            transform=ax.transAxes, fontsize=6.5, va="top", ha="right",
            bbox=dict(boxstyle="round,pad=0.15", fc="white",
                      ec="none", lw=0.3, alpha=0.85))
    ax.set_xticks(range(len(SCENARIOS)))
    ax.set_xticklabels([SSP_LABELS[s] for s in SCENARIOS], fontsize=6)
    ax.set_ylabel(r"$|u_i|/x_i$ during exceedance (%)")
    ax.set_xlim(-0.5, len(SCENARIOS)-0.5)
    ax.set_ylim(0, 30)
    print(f"  (c) Pooled: median={np.median(all_med):.1f}%")
else:
    ax.text(0.5, 0.5, "Feasibility data\nnot available",
            transform=ax.transAxes, ha="center", fontsize=7)
lgrid(ax); plabel(ax, "c")

for ext in ("png","pdf"): fig.savefig(FIG_DIR/f"figS6_feasibility.{ext}")
plt.show()

print("\n" + "="*60)
print("  ALL DONE - saved to", FIG_DIR)
print("  S1: figS1_validation.pdf")
print("  S2: figS2_mass_fidelity.pdf")
print("  S3: figS3_regression.pdf")
print("  S4: figS4_robustness.pdf")
print("  S5: figS5_huc_flexibility.pdf")
print("  S6: figS6_feasibility.pdf")
print("="*60)


In [ ]:
# FIG S7 - Stream-Order Controllability (1x3) [representative member: VIC5/MPI-ESM1-2-HR/SSP2-4.5]
#   (a) Brazos river network colored by Strahler stream order (counts in legend)
#   (b) Effort-reduction Pareto curves for cumulative order subsets (control B restricted to order \u2264 N)
#   (c) Marginal contribution of each stream order to total flood-volume reduction at fixed efforts
# Source: dmd_lqr_stream_order.py outputs:
#   stream_order_pareto_VIC5_MPI-ESM1-2-HR_ssp245.csv  (R-sweep at coarse grid)
#   stream_order_summary_VIC5_MPI-ESM1-2-HR_ssp245.csv (R-sweep at fine grid)
print("\nFIG S7 - Stream-Order Controllability")

from matplotlib.collections import LineCollection
import matplotlib.colors as mcolors

S7_PARETO = "stream_order_pareto_VIC5_MPI-ESM1-2-HR_ssp245.csv"
S7_SUMMARY = "stream_order_summary_VIC5_MPI-ESM1-2-HR_ssp245.csv"

df_so = pd.read_csv(S7_SUMMARY) if Path(S7_SUMMARY).exists() else pd.read_csv(S7_PARETO)
print(f"  Loaded {len(df_so)} rows from {S7_SUMMARY if Path(S7_SUMMARY).exists() else S7_PARETO}")

# Stream-order info from the network shapefile
# Drop reaches with order 0 (disconnected / sentinel rows in NHDPlus); the LQR
# stream-order analysis enumerates Strahler orders 1..7 only.
gdf_so = gdf_net.copy()
gdf_so["StreamOrde"] = gdf_so["StreamOrde"].astype(int)
gdf_so = gdf_so[gdf_so["StreamOrde"] >= 1].copy()
order_counts = gdf_so["StreamOrde"].value_counts().sort_index()
total_n   = int(order_counts.sum())
all_orders = sorted(order_counts.index.tolist())
max_order  = all_orders[-1]
print(f"  Total reaches (order >= 1): {total_n:,}; orders: {order_counts.to_dict()}")

# Use a sequential colormap by order (light = low order, dark = high order)
order_cmap = plt.cm.Purples
order_color = {o: order_cmap(0.25 + 0.65*(i/(len(all_orders)-1)))
               for i, o in enumerate(all_orders)}

fig = plt.figure(figsize=(W_FULL, 75*MM))
gs = fig.add_gridspec(1, 3, left=0.04, right=0.99, top=0.95, bottom=0.16,
                      width_ratios=[1.0, 1.05, 1.05], wspace=0.32)

# ---------- (a) Network colored by stream order ----------
ax = fig.add_subplot(gs[0, 0])
ax.set_aspect("equal"); ax.axis("off")

# Plot in low-to-high order so high-order reaches paint on top with thicker line
order_lw = {o: 0.25 + 0.18*(i+1) for i, o in enumerate(all_orders)}
for o in all_orders:
    sub = gdf_so[gdf_so["StreamOrde"] == o]
    segs = []
    for geom in sub.geometry.values:
        if geom is None or geom.is_empty: continue
        if geom.geom_type == "LineString":
            segs.append(np.asarray(geom.coords))
        elif geom.geom_type == "MultiLineString":
            for g in geom.geoms:
                segs.append(np.asarray(g.coords))
    lc = LineCollection(segs, colors=[order_color[o]], linewidths=order_lw[o],
                        capstyle="round", joinstyle="round",
                        alpha=0.95, zorder=int(o))
    lc.set_rasterized(True); ax.add_collection(lc)

# Frame the basin
b = gdf_so.total_bounds
ax.set_xlim(b[0]-(b[2]-b[0])*0.02, b[2]+(b[2]-b[0])*0.02)
ax.set_ylim(b[1]-(b[3]-b[1])*0.02, b[3]+(b[3]-b[1])*0.02)
add_scalebar(ax, gdf_so)

# Legend showing reach count and percentage
legend_handles = []
for o in all_orders:
    n = int(order_counts.loc[o]); pct = n/total_n*100
    h = plt.Line2D([0],[0], color=order_color[o], lw=1.6+0.25*o,
                    label=f"Order {o} ({n:,}, {pct:.0f}%)")
    legend_handles.append(h)
ax.legend(handles=legend_handles, loc="lower left", fontsize=5.0,
          handlelength=1.4, handletextpad=0.5, labelspacing=0.35,
          frameon=False, title="Stream order (reaches, %)",
          title_fontsize=5.5)
ax.text(-0.02, 1.00, "a", transform=ax.transAxes,
        fontweight="bold", fontsize=11, va="top", ha="right")

# ---------- (b) Pareto curves: control \u2264 N ----------
ax = fig.add_subplot(gs[0, 1])
labels_present = sorted(df_so["constraint_label"].unique(),
                        key=lambda s: int(s.split("_")[-1]))
print(f"  (b) constraints: {labels_present}")

# Cumulative-share table for legend (% of basin reaches at order \u2264 N)
share_at = {}
cum = 0
for o in all_orders:
    cum += int(order_counts.loc[o])
    share_at[o] = cum/total_n*100

eff_grid = np.linspace(0, 15, 200)
for label in labels_present:
    n_thr = int(label.split("_")[-1])
    sub = df_so[df_so["constraint_label"] == label].sort_values("effort_Gm3")
    eff = sub["effort_Gm3"].values
    red = sub["reduction_frac"].values * 100  # to %

    # Force curve to start at (0, 0) for visual continuity
    if eff[0] > 0:
        eff = np.concatenate([[0.0], eff])
        red = np.concatenate([[0.0], red])

    is_all = (n_thr >= max_order)
    color = order_color.get(n_thr, "#333")
    lw = 1.5 if is_all else 0.7
    alpha = 1.0 if is_all else 0.8
    leg = f"All ({share_at[n_thr]:.0f}%)" if is_all else f"\u2264{n_thr} ({share_at[n_thr]:.0f}%)"
    ax.plot(eff, red, color=color, lw=lw, alpha=alpha, label=leg, zorder=2 if is_all else 1)

ax.set_xlim(0, 15)
ax.set_ylim(0, 90)
ax.set_xlabel(r"Effort (Gm$^3$ yr$^{-1}$)")
ax.set_ylabel("Flood volume reduction (%)")
ax.legend(loc="lower right", fontsize=5, title="Max order included",
          title_fontsize=5.5, handlelength=1.4, labelspacing=0.3, frameon=False)
lgrid(ax); plabel(ax, "b")

# ---------- (c) Marginal contribution by order at fixed efforts ----------
ax = fig.add_subplot(gs[0, 2])
fixed_efforts = [1, 3, 5, 10, 15]

# Per-constraint reduction(effort) interpolator
red_funcs = {}
for label in labels_present:
    sub = df_so[df_so["constraint_label"] == label].sort_values("effort_Gm3")
    eff = sub["effort_Gm3"].values
    red = sub["reduction_frac"].values * 100
    if eff[0] > 0:
        eff = np.concatenate([[0.0], eff]); red = np.concatenate([[0.0], red])
    n_thr = int(label.split("_")[-1])
    red_funcs[n_thr] = (eff, red)

def reduction_at(n_thr, E):
    eff, red = red_funcs[n_thr]
    return float(np.interp(E, eff, red))

x_pos = np.arange(len(fixed_efforts))
bw = 0.6

for i, E in enumerate(fixed_efforts):
    bottom = 0.0
    cum_prev = 0.0
    for o in all_orders:
        if o not in red_funcs:
            continue
        red_total = reduction_at(o, E)
        contrib = max(red_total - cum_prev, 0.0)
        if contrib <= 0: cum_prev = red_total; continue
        ax.bar(x_pos[i], contrib, bw, bottom=bottom,
               color=order_color[o], edgecolor="white", linewidth=0.4,
               label=f"Order {o}" if i == 0 else None, zorder=2)
        # In-bar label only for order-1 contribution (matches reference)
        if o == all_orders[0] and contrib > 4.0:
            ax.text(x_pos[i], bottom + contrib/2, f"{contrib:.0f}%",
                    ha="center", va="center", fontsize=5.0,
                    color="white", fontweight="bold")
        bottom += contrib
        cum_prev = red_total
    # Total label above bar
    ax.text(x_pos[i], bottom + 1.5, f"{bottom:.0f}%",
            ha="center", va="bottom", fontsize=5.5, fontweight="bold",
            color="#333")

ax.set_xticks(x_pos)
ax.set_xticklabels([str(e) for e in fixed_efforts])
ax.set_xlabel(r"Effort (Gm$^3$ yr$^{-1}$)")
ax.set_ylabel("Flood volume reduction (%)")
ax.set_ylim(0, 100)
# Build a deduped legend
handles, labels = ax.get_legend_handles_labels()
seen = set(); h2, l2 = [], []
for h, l in zip(handles, labels):
    if l not in seen: seen.add(l); h2.append(h); l2.append(l)
order_pct = {o: int(order_counts.loc[o])/total_n*100 for o in all_orders}
l2 = [f"{l} ({order_pct[int(l.split()[-1])]:.0f}%)" for l in l2]
ax.legend(h2, l2, loc="upper left", fontsize=4.8, ncol=1,
          handlelength=1.0, labelspacing=0.25, frameon=False,
          title="Marginal contribution", title_fontsize=5.2)
lgrid(ax); plabel(ax, "c")

# Reporting
for E in fixed_efforts:
    tot = reduction_at(max_order, E)
    o1  = reduction_at(all_orders[0], E)
    print(f"  (c) E={E:>2.0f} Gm3/yr: total={tot:5.1f}%, order-1 marginal={o1:5.1f}%")

for ext in ("png","pdf"): fig.savefig(FIG_DIR/f"figS7_stream_order.{ext}")
plt.show()
print("  Saved figS7_stream_order")
